<center><h1  style="color:white; background-color:#000000; border-radius: 0px; padding:25px;"> Fairness </h1></center>

This notebook illustrates fairness modeule on iris data set.

In [ ]:
import os

os.chdir("../")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from sklearn.metrics import average_precision_score, accuracy_score, roc_auc_score, confusion_matrix

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from woodtapper.fairness import OTFairBoostClassifier

## Evaluation and metric functions:

In [ ]:
def _safe_div(num: float, den: float) -> float:
    """Safe division helper (returns NaN if denominator is zero)."""
    return float(num / den) if den != 0 else np.nan

def _binary_rates(y_true, y_preds):
    """
    Compute classic binary classification rates.

    Important: y_true and y_pred must be 0/1 encoded (positive class = 1).

    Returns:
        - TPR (recall / sensitivity)
        - FPR
        - PPV (precision)
        - PPR (predicted positive rate) -> used for DP/DI
    """
    cm = confusion_matrix(y_true, y_preds, labels=[0, 1])
    # cm is guaranteed to be 2x2 with labels=[0,1]
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

    tpr = _safe_div(tp, tp + fn)
    fpr = _safe_div(fp, fp + tn)
    ppv = _safe_div(tp, tp + fp)
    ppr = float(np.mean(y_preds)) if y_preds.size else np.nan

    return {"TPR": tpr, "FPR": fpr, "PPV": ppv, "PPR": ppr}

def _worst_pairwise_gap(values):
    vals = [v for v in values.values() if v is not None and not np.isnan(v)]
    if len(vals) < 2:
        return np.nan
    return float(np.max(vals) - np.min(vals))

def _worst_pairwise_di(values):
    vals = [v for v in values.values() if v is not None and not np.isnan(v)]
    vals = [v for v in vals if v > 0]  # if zero exists -> DI collapses to 0

    if len(vals) < 2:
        return np.nan

    return float(np.min(vals) / np.max(vals))

In [ ]:
def derive_fairness_metrics(y_true,y_preds,sensitive_attribute,min_group_n=10):
    groups = pd.Series(sensitive_attribute).astype(str)
    rows = []

    ppr_by_g = {}
    tpr_by_g = {}
    fpr_by_g = {}
    ppv_by_g = {}

    for g in groups.unique():
        idx = (groups == g).to_numpy()
        n_g = int(idx.sum())

        if n_g < min_group_n:
            rows.append({"group": g, "n": n_g, "PPR": np.nan, "TPR": np.nan, "FPR": np.nan, "PPV": np.nan})
            continue

        rates = _binary_rates(y_true[idx].astype(int), y_preds[idx].astype(int))
        ppr_by_g[g] = rates["PPR"]
        tpr_by_g[g] = rates["TPR"]
        fpr_by_g[g] = rates["FPR"]
        ppv_by_g[g] = rates["PPV"]

        rows.append({"group": g, "n": n_g, **rates})

    rates_df = pd.DataFrame(rows)

    dp_gap = _worst_pairwise_gap(ppr_by_g)
    di = _worst_pairwise_di(ppr_by_g)
    eopp_gap = _worst_pairwise_gap(tpr_by_g)
    fpr_gap = _worst_pairwise_gap(fpr_by_g)
    ppv_gap = _worst_pairwise_gap(ppv_by_g)
    
    return dp_gap, di, eopp_gap, fpr_gap, ppv_gap, rates_df



def make_stratify_labels(y, sensitive_attribute, stratify_on_joint=True):
    """
    Build stratification labels for CV.

    If stratify_on_joint=True, stratify on joint labels (y, a) to stabilize
    subgroup proportions across folds. If some joint labels are too rare,
    fallback to stratifying on y only.
    """
    if isinstance(y, pd.Series):
        y_arr = y.to_numpy()
    else:
        y_arr = y

    if not stratify_on_joint:
        return y_arr

    if isinstance(sensitive_attribute, pd.Series):
        joint = (y.astype(str) + "__" + sensitive_attribute.astype(str)).to_numpy()
    else:
        joint = (y.astype(str) + "__" + sensitive_attribute.astype(str))#.to_numpy()

    # If very rare joint labels exist, stratification can fail; fallback to y-only
    _, counts = np.unique(joint, return_counts=True)
    if counts.size == 0 or np.min(counts) < 2:
        return y_arr

    return joint

## Load data :

In [ ]:
adult = fetch_openml("adult", version=2)
X_df = adult.data.drop(columns=["sex"])
numeric_cols = X_df.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_df.columns if c not in numeric_cols]
X = X_df#.to_numpy()
y = adult.target
sensitive_attribute = adult.data[["sex"]]

In [ ]:
sensitive_attribute = adult.data["sex"]
sensitive_attribute = sensitive_attribute.map({"Male": 0, "Female": 1})
#sensitive_attribute = sensitive_attribute.to_numpy()
sensitive_attribute = pd.DataFrame(sensitive_attribute, dtype="int")
sensitive_attribute

In [ ]:
y = adult.target
y = y.map({"<=50K": 0, ">50K": 1})
#y = y.to_numpy()
y = pd.DataFrame(y, dtype="int")
y

In [ ]:
X_train, X_test, y_train, y_test, sensitive_attribute_train, sensitive_attribute_test = train_test_split(X, y, sensitive_attribute, random_state=0, test_size=0.2,
                                                                                                          stratify=make_stratify_labels(y, sensitive_attribute, stratify_on_joint=True))

## OTFairBoostClassifier

In [ ]:
ot_clf = OTFairBoostClassifier(
    n_estimators=1000,
    lambda_fairness_value=1.0,
    fairness_mode="Demographic_Parity", #Demographic_Parity Equality_of_Odds
    verbosity=-1,
    learning_rate=0.1,
    num_leaves=64,
    min_child_samples=50,
    colsample_bytree=0.9,
    subsample=0.9,
    subsample_freq=1,
    random_state=42,
    n_jobs=8,
)


num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

ot_model_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("ot_clf", ot_clf)])
ot_model_pipeline.fit(X_train, y_train, ot_clf__sensitive_attribute=sensitive_attribute_train)

In [ ]:
X_test

In [ ]:
y_preds = ot_model_pipeline.predict(X_test)
y_preds_probas = ot_model_pipeline.predict_proba(X_test)

In [ ]:
pr_auc = average_precision_score(y_true=y_test, y_score=y_preds_probas[:, 1])
roc_auc = roc_auc_score(y_true=y_test, y_score=y_preds_probas[:, 1])
accuracy = accuracy_score(y_true=y_test, y_pred=y_preds)
print("### Predictive Perfs ###")
print("PR AUC:", pr_auc)
print("ROC AUC:", roc_auc)
print("Accuracy:", accuracy)

dp_gap, di, eopp_gap, fpr_gap, ppv_gap, rates_df=derive_fairness_metrics(y_true=y_test.to_numpy().ravel(), 
                                                                         y_preds=y_preds,
                                                                         sensitive_attribute=sensitive_attribute_test.to_numpy().ravel(), min_group_n=10)
print("Fairness Perfs")
print("DP GAP:", dp_gap)
print("DI:", di)
print("EOPP GAP:", eopp_gap)